In [3]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
ground_truth[10]

{'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
 'document': '489dd1c9d9'}

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [6]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [7]:
q = ground_truth[10]
q

{'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
 'document': '489dd1c9d9'}

In [8]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [9]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [11]:
q['question']

'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?'

In [12]:
answer = assistant.rag(q['question'])

In [13]:
assistant.total_cost()

0.0009854999999999998

In [14]:
print(answer)

The Zoom link isn’t shared with students. Instead, you can join the office hours or live workshop via:

- **YouTube Live** on the **DataTalksClub YouTube channel**
- **Announcements on Telegram or Slack**, where the live video URL is posted before it starts
- **Slido** for questions, with the link pinned in the live chat

Don’t post questions in the chat, since they may get missed if the session is busy.


In [15]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [16]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
 'answer_llm': 'The Zoom link isn’t shared with students. Instead, you can join the office hours or live workshop via:\n\n- **YouTube Live** on the **DataTalksClub YouTube channel**\n- **Announcements on Telegram or Slack**, where the live video URL is posted before it starts\n- **Slido** for questions, with the link pinned in the live chat\n\nDon’t post questions in the chat, since they may get missed if the session is busy.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as t

In [17]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [18]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
 'answer_llm': "You can’t join through a shared Zoom link as a student. The Zoom link is only published to instructors, presenters, and TAs.\n\nFor students, the live session is available via:\n\n- **YouTube Live** on the **DataTalksClub YouTube channel**\n- The stream/video URL is posted in the **announcements channel on Telegram and Slack** before it starts\n- Questions should be submitted through **Slido**; the link is pinned in the chat when the session is live\n\nIf the session is not announced or you can’t find the link, I don't know.",
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live 

In [19]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the office hours or live workshop if the Zoom link isn’t shared with students?',
 'answer_llm': 'You can still join the office hours or live workshop even if the Zoom link isn’t shared with students:\n\n- **Watch via YouTube Live** on the **DataTalksClub YouTube channel**\n- Check the **announcements channel on Telegram and Slack** for the live video URL before it starts\n- Ask questions through **Slido** when the session is live (the Slido link is pinned in the chat)\n\nDon’t post questions in the chat, since they may be missed.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t po

In [20]:
assistant.total_cost()

0.0030914999999999996

In [21]:
assistant.reset_usage()

In [22]:
assistant.total_cost()

0.0

In [23]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [27]:
from openai import RateLimitError
from tenacity import retry, wait_random_exponential, stop_after_attempt, retry_if_exception_type

@retry(
    retry=retry_if_exception_type(RateLimitError),
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(6),
)
def _call_rag(question):
    return assistant.rag(question)


def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = _call_rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [28]:
with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/590 [00:00<?, ?it/s]

In [29]:
results[:10]

[{'question': 'Can I still join the course if I found it late?',
  'answer_llm': 'Yes, you can still join the course if you found it late. If you want to receive a certificate, make sure you submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course after it has already begun?',
  'answer_llm': 'Yes. You can start the course whenever you want, even after it has already begun. The videos and GitHub materials are available, and you can follow the weekly workflow at your own pace.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join now, will I still be able to get a certificate?',
  'answer_llm': 'Yes, but only if you join while 

In [30]:
df_results = pd.DataFrame(results)

In [31]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Can I still join the course if I found it late?,"Yes, you can still join the course if you foun...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Is it okay to start the course after it has al...,Yes. You can start the course whenever you wan...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join now, will I still be able to get a c...","Yes, but only if you join while the course is ...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes. To get the certificate, you need to submi...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,What happens if I join the course after projec...,If you join after project submissions are no l...,"Yes, but if you want to receive a certificate,...",74eb249bbf


In [32]:
assistant.total_cost()

1.2856515000000015

In [33]:
df_results.to_csv("data/rag-answers-new.csv", index=False)